In [3]:
import json 
from pathlib import Path
from pprint import pprint


def compare_json_files(path_a, path_b):
    path_a = Path(path_a)
    path_b = Path(path_b)

    with path_a.open("r", encoding="utf-8") as f:
        data_a = json.load(f)
    with path_b.open("r", encoding="utf-8") as f:
        data_b = json.load(f)

    same = data_a == data_b
    print(f"Same content: {same}")

    if same:
        return True

    print(f"\nFiles differ:\n- {path_a}\n- {path_b}")
    print(f"Type A: {type(data_a).__name__}")
    print(f"Type B: {type(data_b).__name__}")

    if isinstance(data_a, dict) and isinstance(data_b, dict):
        only_a = sorted(set(data_a) - set(data_b))
        only_b = sorted(set(data_b) - set(data_a))
        common_diff = sorted(k for k in set(data_a) & set(data_b) if data_a[k] != data_b[k])

        print(f"\nKeys only in first: {only_a[:20]}")
        print(f"Keys only in second: {only_b[:20]}")
        print(f"Keys with different values: {common_diff[:20]}")

        if common_diff:
            key = common_diff[0]
            print(f"\nExample difference for key: {key}")
            print("First:")
            pprint(data_a[key])
            print("Second:")
            pprint(data_b[key])

    return False


root = "/root/projects/O-LoRA/CL_project/CL_Benchmark/"
compare_json_files(
    f"{root}BoolQA/BoolQA/dev.json",
    f"{root}BoolQA/BoolQA/test.json",
)

Same content: True


True

In [ ]:
from collections import defaultdict
import random
from pathlib import Path
import json


src_root = Path("/root/projects/O-LoRA/CL_project/CL_Benchmark")
dst_root = Path("/root/projects/O-LoRA/CL_project/CL_Benchmark_dev")
seed = 42
train_ratio = 0.8
test_cap_per_class = 500


dataset_dirs = {
    "MNLI": "NLI/MNLI",
    "CB": "NLI/CB",
    "WiC": "WiC/WiC",
    "COPA": "COPA/COPA",
    "QQP": "QQP/QQP",
    "BoolQA": "BoolQA/BoolQA",
    "RTE": "NLI/RTE",
    "IMDB": "SC/IMDB",
    "yelp": "SC/yelp",
    "amazon": "SC/amazon",
    "SST-2": "SC/SST-2",
    "dbpedia": "TC/dbpedia",
    "agnews": "TC/agnews",
    "MultiRC": "MultiRC/MultiRC",
    "yahoo": "TC/yahoo",
}


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)


def stratified_train_val_split(data, train_ratio=0.8, seed=42):
    rng = random.Random(seed)
    by_label = defaultdict(list)
    for item in data:
        by_label[item["label"]].append(item)

    train_split = []
    val_split = []
    for label, items in by_label.items():
        items = items[:]
        rng.shuffle(items)
        split_idx = int(len(items) * train_ratio)
        if len(items) > 1:
            split_idx = min(max(split_idx, 1), len(items) - 1)
        train_split.extend(items[:split_idx])
        val_split.extend(items[split_idx:])

    rng.shuffle(train_split)
    rng.shuffle(val_split)
    return train_split, val_split


def cap_test_per_class(data, max_per_class=500, seed=42):
    rng = random.Random(seed)
    by_label = defaultdict(list)
    for item in data:
        by_label[item["label"]].append(item)

    capped = []
    for label, items in by_label.items():
        items = items[:]
        rng.shuffle(items)
        capped.extend(items[:max_per_class])

    rng.shuffle(capped)
    return capped


summary = {}

for dataset_name, dataset_dir in dataset_dirs.items():
    src_dir = src_root / dataset_dir
    dst_dir = dst_root / dataset_dir

    train_data = load_json(src_dir / "train.json")
    test_data = load_json(src_dir / "test.json")
    labels = load_json(src_dir / "labels.json")

    new_train, new_val = stratified_train_val_split(train_data, train_ratio=train_ratio, seed=seed)
    new_test = cap_test_per_class(test_data, max_per_class=test_cap_per_class, seed=seed)

    save_json(dst_dir / "train.json", new_train)
    save_json(dst_dir / "dev.json", new_val)   # validation split saved as dev.json
    save_json(dst_dir / "test.json", new_test)
    save_json(dst_dir / "labels.json", labels)

    summary[dataset_name] = {
        "train": len(new_train),
        "dev": len(new_val),
        "test": len(new_test),
    }

print("Wrote rebuilt datasets to:", dst_root)
print("\nSummary")
print("-" * 40)
for dataset_name, sizes in summary.items():
    print(
        f"{dataset_name:10} -> train={sizes['train']}, dev={sizes['dev']}, test={sizes['test']}"
    )

Wrote rebuilt datasets to: /root/projects/O-LoRA/CL_project/CL_Benchmark_dev

Summary
----------------------------------------
MNLI       -> train=2400, dev=600, test=1631
CB         -> train=199, dev=51, test=56
WiC        -> train=1599, dev=401, test=638
COPA       -> train=320, dev=80, test=100
QQP        -> train=1599, dev=401, test=1000
BoolQA     -> train=1599, dev=401, test=1000
RTE        -> train=1599, dev=401, test=277
IMDB       -> train=1599, dev=401, test=1000
yelp       -> train=3997, dev=1003, test=2500
amazon     -> train=3998, dev=1002, test=2500
SST-2      -> train=1599, dev=401, test=872
dbpedia    -> train=11194, dev=2806, test=7000
agnews     -> train=3199, dev=801, test=2000
MultiRC    -> train=1599, dev=401, test=1000
yahoo      -> train=7997, dev=2003, test=5000



Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



In [2]:
from collections import Counter
import json


dataset_dirs = {
    "MNLI": "NLI/MNLI",
    "CB": "NLI/CB",
    "WiC": "WiC/WiC",
    "COPA": "COPA/COPA",
    "QQP": "QQP/QQP",
    "BoolQA": "BoolQA/BoolQA",
    "RTE": "NLI/RTE",
    "IMDB": "SC/IMDB",
    "yelp": "SC/yelp",
    "amazon": "SC/amazon",
    "SST-2": "SC/SST-2",
    "dbpedia": "TC/dbpedia",
    "agnews": "TC/agnews",
    "MultiRC": "MultiRC/MultiRC",
    "yahoo": "TC/yahoo",
}


def count_samples_per_label(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return Counter(item["label"] for item in data)


results = {}

root = "/root/projects/O-LoRA/CL_project/CL_Benchmark_dev/"
for dataset_name, dataset_dir in dataset_dirs.items():
    results[dataset_name] = {
        "train": count_samples_per_label(f"{root}{dataset_dir}/train.json"),
        "dev": count_samples_per_label(f"{root}{dataset_dir}/dev.json"),
        "test": count_samples_per_label(f"{root}{dataset_dir}/test.json"),
    }

for dataset_name, split_counts in results.items():
    print(f"\n{'=' * 20} {dataset_name} {'=' * 20}")
    for split_name, counts in split_counts.items():
        print(f"{split_name}:")
        for label, count in sorted(counts.items()):
            print(f"  {label}: {count}")


==================== MNLI ====================
train:
  contradiction: 808
  entailment: 796
  neutral: 796
dev:
  contradiction: 202
  entailment: 199
  neutral: 199
test:
  -: 131
  contradiction: 500
  entailment: 500
  neutral: 500

==================== CB ====================
train:
  contradiction: 95
  entailment: 92
  neutral: 12
dev:
  contradiction: 24
  entailment: 23
  neutral: 4
test:
  contradiction: 28
  entailment: 23
  neutral: 5

==================== WiC ====================
train:
  False: 810
  True: 789
dev:
  False: 203
  True: 198
test:
  False: 319
  True: 319

==================== COPA ====================
train:
  A: 156
  B: 164
dev:
  A: 39
  B: 41
test:
  A: 55
  B: 45

==================== QQP ====================
train:
  False: 991
  True: 608
dev:
  False: 248
  True: 153
test:
  False: 500
  True: 500

==================== BoolQA ====================
train:
  False: 607
  True: 992
dev:
  False: 152
  True: 249
test:
  False: 500
  True: 500

========